# Processing and Curation - History & Biography

This notebook executes the curation pipeline for History & Biography and produces book and interaction level Parquet datasets.

## Cleaning Decisions

- Numerical metadata arrives as strings and is converted using `errors='coerce'`.
- Nested fields (`authors`, `popular_shelves`, `series`, `similar_books`) are summarized into modelable columns.
- Reviews are cleaned of basic HTML and redundant spaces.
- Missing ratings are separated from explicit ratings using `rating_missing` and `rating_clean`.
- Outliers are documented and p99 capped features are generated without modifying the original values.

In [ ]:
from pathlib import Path
import os, sys, json
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.process_goodreads import process_category
from src.config import CATEGORIES

cfg = CATEGORIES['history_biography']

In [ ]:
report = process_category('history_biography', chunksize=100_000, bucket_count=256)
report

## Validación de salidas

Las validaciones comprueban lectura Parquet, llaves no nulas, ratings limpios y deduplicación básica.

In [ ]:
books = pd.read_parquet(cfg.processed_dir / 'books_curated.parquet')
interactions = pd.read_parquet(cfg.processed_dir / 'interactions_curated.parquet')

assert books['book_id'].notna().all()
assert interactions['book_id'].notna().all()
assert interactions['rating_clean'].dropna().between(1, 5).all()
assert books['book_id'].duplicated().sum() == 0
assert interactions['review_id'].dropna().duplicated().sum() == 0

books.head(), interactions.head()